# AML Patient Eligibility

Split from `Patient Eligibility.ipynb`: setup, Rx/Mx exact-date activity tables, eligibility flags and the final eligibility flags table. Fully commented-out cells removed.

Reads the LoT build outputs (combined LoT table, episode table, `PATIENT_INTENSITY_BASED_ON_AGE_N_PRODUCT`), so run it after the LoT notebook.

# SPARK SESSION

In [ ]:
import findspark
import pandas as pd
findspark.init ()
import pyspark
from pyspark.sql import SparkSession

spark= SparkSession.builder\
              .config('spark.executor.instances','15')\
              .config('spark.driver.memory','5g')\
              .config('spark.executor.memory','12g')\
              .config('spark.executor.cores','5')\
              .config('spark.executor.memoryOverhead','4g')\
              .appName("AML_KA")\
              .master("yarn")\
              .getOrCreate()

In [ ]:
spark.sparkContext.addPyFile("pldlib.egg")
from pldlib import stencil

# MAX MONTH CALCULATION

In [ ]:
backup_dt = spark.sql('''
SELECT MAX(TO_DATE(CLAIM_DATE)) 
FROM abv_val_ptd_onc_venc_synd.all_claims_combined_fact_vw
where source_type = 'RX FACT'
and source_flag = 'SHA_PTD'
and source_market = 'ONC'
and market_code = 'VENC_AML'
''')

In [ ]:
new_data_date = spark.sql('''
SELECT MAX(TO_DATE(CLAIM_DATE)) 
FROM abv_val_ptd_onc_venc_synd.all_claims_combined_fact_vw
where source_type = 'RX FACT'
and source_flag = 'SHA_PTD'
and source_market = 'ONC'
and market_code = 'VENC_AML'
''')

In [ ]:
from datetime import datetime
from dateutil.relativedelta import relativedelta  

# Convert backup_dt to pandas and get backup_max_dt and month
backup_max_dt = backup_dt.toPandas().iat[0, 0].strftime("%Y-%m-%d")
month = backup_dt.toPandas().iat[0, 0].strftime("%b_%y").upper() 

# Get the new maximum date from Spark DataFrame
new_max_dt_row = new_data_date.first()  # This is a Row object
new_max_dt = new_max_dt_row[0]  # Access the first column to get the date

# Since new_max_dt is a date object, we can directly work with it
if isinstance(new_max_dt, datetime):
    # If it's already a datetime object, proceed directly
    res = new_max_dt + relativedelta(months=-6) + relativedelta(days=1)
else:
    # If it's a date object, you can cast it to datetime
    res = datetime.combine(new_max_dt, datetime.min.time()) + relativedelta(months=-6) + relativedelta(days=1)

currsem = int((res.month - 1) / 6 + 1)
exc_dt = datetime(res.year, 6 * currsem - 5, 1).strftime("%Y-%m-%d")

# Optionally print results for debugging
print(f"Backup Max Date: {backup_max_dt}")
print(f"Month: {month}")
print(f"New Max Date: {new_max_dt}")
print(f"Resulting Date (res): {res}")
print(f"Exception Date (exc_dt): {exc_dt}")


In [ ]:
print("Max date (backup): ", backup_max_dt)
print("Max month (backup): ", month)

print("Max date (new data): ", new_max_dt)
print("Exclusion sem for elig: ", exc_dt)

In [ ]:
month

# GLOBAL VARIABLES

In [ ]:
cws = 'Z_ABV_CWS_MABI_ONC_ANALYTICS' 

# HCP-CHILD
reltio_tbl = 'abv_mhcd_synd.mhcd_rltn_all_org_customr_wkly_tbl'
heme_account='{cws}.heme_account'.format(cws=cws) # UPLOADED BY COE

tx_tbl_name = '{cws}.MABI_AML_REGIMEN_REVAMP_UPDATED_TX_TBL_BUSINESS_RULE_CHANGE_VAL'.format(cws=cws)
tx_tbl_final_name = '{cws}.MABI_AML_REGIMEN_REVAMP_UPDATED_TX_TBL_FINAL_BUSINESS_RULE_CHANGE_VAL'.format(cws=cws)

sob_tbl_name = '{cws}.MABI_AML_REGIMEN_REVAMP_SOB_BUSINESS_RULE_CHANGE_VAL'.format(cws=cws)
episode_tbl_name = '{cws}.MABI_AML_REGIMEN_REVAMP_EPISODE_BUSINESS_RULE_CHANGE_VAL'.format(cws=cws)
regimen_tbl_name = '{cws}.MABI_AML_REGIMEN_REVAMP_BASIC_REGIMEN_BUSINESS_RULE_CHANGE_VAL'.format(cws=cws)

patient_group = '{cws}.MABI_AML_REGIMEN_REVAMP_PATIENT_COHORT_BUSINESS_RULE_CHANGE_VAL'.format(cws=cws)

# LoT IC INELIGIBLE
ic_inelig_final = '{cws}.MABI_AML_IC_INELIG_LOT_FINAL_BUSINESS_RULE_CHANGE_VAL'.format(cws=cws)

# LOT IC ELIGIBLE
ic_elig_final = '{cws}.MABI_AML_IC_ELIG_LOT_FINAL_BUSINESS_RULE_CHANGE_VAL'.format(cws=cws) # LOT table IC ELIG
pat_comb_lot = '{cws}.MABI_AML_PATS_COMB_LOT_TBL_BUSINESS_RULE_CHANGE_VAL'.format(cws=cws)  # UNION OF BOTH LOT FINAL TABLES
pat_comb_lot_rr = '{cws}.MABI_AML_PATS_LOT_RR_12M_UPD_BUSINESS_RULE_CHANGE_VAL'.format(cws=cws)    

pat_fst_dx_tx = '{cws}.MABI_AML_LOT_PAT_FST_DX_TX_TBL_BUSINESS_RULE_CHANGE_VAL'.format(cws=cws)

dx_lookback_elig = '{cws}.MABI_AML_LOT_PAT_DX_LB_ELIG_TBL_BUSINESS_RULE_CHANGE_VAL'.format(cws=cws)
tx_lookback_elig = '{cws}.MABI_AML_LOT_PAT_TX_MX_LB_ELIG_TBL_BUSINESS_RULE_CHANGE_VAL'.format(cws=cws)
tx_lookback_elig_rx = '{cws}.MABI_AML_LOT_PAT_TX_RX_LB_ELIG_TBL_BUSINESS_RULE_CHANGE_VAL'.format(cws=cws)
tx_txl_mx_elig = '{cws}.MABI_AML_LOT_PAT_TX_MX_TXL_ELIG_TBL_BUSINESS_RULE_CHANGE_VAL'.format(cws=cws)
tx_txl_rx_elig = '{cws}.MABI_AML_LOT_PAT_TX_RX_TXL_ELIG_TBL_BUSINESS_RULE_CHANGE_VAL'.format(cws=cws)

tx_lookback_elig_ven = '{cws}.MABI_AML_LOT_PAT_VEN_TX_MX_LB_ELIG_TBL_BUSINESS_RULE_CHANGE_VAL'.format(cws=cws)
tx_lookback_elig_rx_ven = '{cws}.MABI_AML_LOT_PAT_VEN_TX_RX_LB_ELIG_TBL_BUSINESS_RULE_CHANGE_VAL'.format(cws=cws)
tx_txl_mx_elig_ven = '{cws}.MABI_AML_LOT_PAT_VEN_TX_MX_TXL_ELIG_TBL_BUSINESS_RULE_CHANGE_VAL'.format(cws=cws)
tx_txl_rx_elig_ven = '{cws}.MABI_AML_LOT_PAT_VEN_TX_RX_TXL_ELIG_TBL_BUSINESS_RULE_CHANGE_VAL'.format(cws=cws)

#HCP AFFILIATIONS
hcp_affiliations = '{cws}.MABI_AML_HCP_AFFILIATIONS'.format(cws=cws)

# LoT TABLE ELIGIBILITY CHECK
elig_flags_final='{cws}.MABI_AML_LOT_PAT_ELIG_FLAGS_FINAL_BUSINESS_RULE_CHANGE_VAL'.format(cws=cws)

#RR AML DX ONLY
aml_dx = ('C92.01', 'C92.02', 'C92.51', 'C92.52', 'C92.61', 'C92.62', 'C92.A1', 'C92.A2', 'C93.01', 'C93.02', 'C94.01', 'C94.21', 'C94.22')

#Product codes for Onureg, Inqovi, Rezlidhia and Vanflyta 

# onureg_product_code =  spark.sql('''
#                                         select distinct source_product_ndc_code as PRODUCT_NDC_CODE
#           from abv_val_ptd_onc_synd.dim_product_vw
#           where source_market = 'ONC'
#           and source_flag = 'SHA_PTD'
#           and UPPER(SOURCE_DRUG_GENERIC_NAME) LIKE ('%AZACITIDINE%') 
#           AND SOURCE_DRUG_FORM IN ('TAB','CAP')
#                                         --SELECT DISTINCT PRODUCT_NDC_CODE
#                                         --FROM SHA_PTD_ONC_SYND.PTD_DRUG_TBL
#                                         --WHERE UPPER(DRUG_GENERIC_NAME) LIKE ('%AZACITIDINE%') AND DRUG_FORM IN ('TAB','CAP')
#                                         UNION ALL
#                                  select distinct procedure_code as PRODUCT_NDC_CODE
#           from abv_val_ptd_onc_synd.px_fact_curated_vw
#           where source_market = 'ONC'
#           and source_flag = 'SHA_PTD'
#           and UPPER(PROCEDURE_DESCription) LIKE ('%ONUREG%')
#                                         --SELECT DISTINCT PROCEDURE_CODE AS PRODUCT_NDC_CODE
#                                         --FROM SHA_PTD_ONC_SYND.PTD_PROCEDURE_CODE_TBL
#                                         --WHERE UPPER(PROCEDURE_DESC) LIKE ('%ONUREG%')
#                                  ''') 
# df1 = onureg_product_code.toPandas()
# df2 = df1['PRODUCT_NDC_CODE'].values.tolist()
# df3 = list(df2)
# onureg_codes = str(df3)[1:-1]

# inqovi_product_code =  spark.sql('''
#                                         select distinct source_product_ndc_code as PRODUCT_NDC_CODE
#           from abv_val_ptd_onc_synd.dim_product_vw
#           where source_market = 'ONC'
#           and source_flag = 'SHA_PTD'
#           and UPPER(SOURCE_DRUG_GENERIC_NAME) LIKE ('%DECITABINE/CEDAZURIDINE%') 
#           AND SOURCE_DRUG_FORM IN ('TAB','CAP')        
                                 
#                                         --SELECT DISTINCT PRODUCT_NDC_CODE 
#                                         --FROM SHA_PTD_ONC_SYND.PTD_DRUG_TBL
#                                         --WHERE UPPER(DRUG_GENERIC_NAME) LIKE ('%DECITABINE/CEDAZURIDINE%') AND DRUG_FORM IN ('TAB','CAP')
#                                         UNION ALL
#                                         select distinct procedure_code as PRODUCT_NDC_CODE
#           from abv_val_ptd_onc_synd.px_fact_curated_vw
#           where source_market = 'ONC'
#           and source_flag = 'SHA_PTD'
#           and (UPPER(PROCEDURE_DESCription) LIKE ('%INQOVI%') or UPPER(PROCEDURE_DESCription) LIKE ('%DECITABINE/CEDAZURIDINE%'))
                                 
#                                         --SELECT DISTINCT PROCEDURE_CODE AS PRODUCT_NDC_CODE
#                                         --FROM SHA_PTD_ONC_SYND.PTD_PROCEDURE_CODE_TBL
#                                         --WHERE UPPER(PROCEDURE_DESC) LIKE ('%INQOVI%') OR UPPER(PROCEDURE_DESC) LIKE ('%DECITABINE/CEDAZURIDINE%')
#                                 ''') 
# df4 = inqovi_product_code.toPandas()
# df5 = df4['PRODUCT_NDC_CODE'].values.tolist()
# df6 = list(df5)
# inqovi_codes = str(df6)[1:-1]

# rezlidhia_product_code =  spark.sql('''
#                                         select distinct source_product_ndc_code as PRODUCT_NDC_CODE
#           from abv_val_ptd_onc_synd.dim_product_vw
        #   where source_market = 'ONC'
        #   and source_flag = 'SHA_PTD'
        #   and (UPPER(SOURCE_DRUG_GENERIC_NAME) LIKE ('%OLUTASIDENIB%') or UPPER(SOURCE_DRUG_GENERIC_NAME) LIKE ('%REZLIDHIA%'))

        #                                 --SELECT DISTINCT PRODUCT_NDC_CODE 
        #                                 --FROM SHA_PTD_ONC_SYND.PTD_DRUG_TBL
        #                                 --WHERE UPPER(DRUG_GENERIC_NAME) LIKE ('%OLUTASIDENIB%') OR UPPER(DRUG_GENERIC_NAME) LIKE ('%REZLIDHIA%') 
        #                                 UNION ALL
        #                             select distinct procedure_code as PRODUCT_NDC_CODE
        #   from abv_val_ptd_onc_synd.px_fact_curated_vw
        #   where source_market = 'ONC'
        #   and source_flag = 'SHA_PTD'
        #   and (UPPER(PROCEDURE_DESCription) LIKE ('%REZLIDHIA%') or UPPER(PROCEDURE_DESCription) LIKE ('%OLUTASIDENIB%'))


        #                                 --SELECT DISTINCT PROCEDURE_CODE AS PRODUCT_NDC_CODE
        #                                 --FROM SHA_PTD_ONC_SYND.PTD_PROCEDURE_CODE_TBL
        #                                 --WHERE UPPER(PROCEDURE_DESC) LIKE ('%REZLIDHIA%') OR UPPER(PROCEDURE_DESC) LIKE ('%OLUTASIDENIB%')
#                                 ''') 
# df7 = rezlidhia_product_code.toPandas()
# df8 = df7['PRODUCT_NDC_CODE'].values.tolist()
# df9 = list(df8)
# rezlidhia_codes = str(df9)[1:-1]

# vanflyta_product_code =  spark.sql('''  
#                                    select distinct source_product_ndc_code as PRODUCT_NDC_CODE
#           from abv_val_ptd_onc_synd.dim_product_vw
#           where source_market = 'ONC'
#           and source_flag = 'SHA_PTD'
#           and (UPPER(SOURCE_DRUG_GENERIC_NAME) LIKE ('%QUIZARTINIB%') or UPPER(SOURCE_DRUG_GENERIC_NAME) LIKE ('%VANFLYTA%'))


#                                         --SELECT DISTINCT PRODUCT_NDC_CODE 
#                                         --FROM SHA_PTD_ONC_SYND.PTD_DRUG_TBL
#                                         --WHERE UPPER(DRUG_GENERIC_NAME) LIKE ('%QUIZARTINIB%') OR UPPER(DRUG_GENERIC_NAME) LIKE ('%VANFLYTA%') 
#                                         UNION ALL
#                                    select distinct procedure_code as PRODUCT_NDC_CODE
#           from abv_val_ptd_onc_synd.px_fact_curated_vw
#           where source_market = 'ONC'
#           and source_flag = 'SHA_PTD'
#           and (UPPER(PROCEDURE_DESCription) LIKE ('%VANFLYTA%') or UPPER(PROCEDURE_DESCription) LIKE ('%QUIZARTINIB%'))


#                                         --SELECT DISTINCT PROCEDURE_CODE AS PRODUCT_NDC_CODE
#                                         --FROM SHA_PTD_ONC_SYND.PTD_PROCEDURE_CODE_TBL
#                                         --WHERE UPPER(PROCEDURE_DESC) LIKE ('%VANFLYTA%') OR UPPER(PROCEDURE_DESC) LIKE ('%QUIZARTINIB%')
#                                 ''') 
# df10 = vanflyta_product_code.toPandas()
# df11 = df10['PRODUCT_NDC_CODE'].values.tolist()
# df12 = list(df11)
# vanflyta_codes = str(df12)[1:-1]


In [ ]:
import sys
sys.path.append('../../config')
from load_config import load_config
import intensity_definition as idef

icfg = load_config('eligibility.yaml')
INTENSITY_DEFINITION = idef.definition(icfg)     # 'legacy' (default) or 'new'
DEF_TAG = idef.tag(icfg)
print('IC eligible / ineligible definition:', idef.label(icfg))

# Rule B (new-definition) class table, as before for legacy runs.
rule_b_tbl = 'Z_ABV_CWS_MABI_ONC_ANALYTICS.PATIENT_INTENSITY_BASED_ON_AGE_N_PRODUCT'
if INTENSITY_DEFINITION == 'new':
    # Read the new-definition LoT build (AML_LOT_PATIENT_INTENSITY_LOT run with intensity_definition: new)
    # and write every eligibility output with the tag, so legacy flag tables are not overwritten.
    # The combined LoT TYPE then carries the new class, so PATIENT_COHORT = new definition.
    lot_cfg = load_config('intensity_lot.yaml')
    pat_comb_lot = lot_cfg['tables']['pat_comb_lot'] + DEF_TAG
    patient_group = lot_cfg['tables']['patient_group'] + DEF_TAG
    rule_b_tbl = lot_cfg['tables']['patient_intensity_based_on_age_n_product'] + DEF_TAG
    for _v in idef.ELIGIBILITY_DEPENDENT_TABLES:
        globals()[_v] = globals()[_v] + DEF_TAG
print('Combined LoT read:', pat_comb_lot, '| flags written to:', elig_flags_final)

Creating Rx and Mx Activity Tables (Exact Dates) for Patient Eligibility

In [ ]:

spark.sql(''' DROP TABLE IF EXISTS Z_ABV_CWS_MABI_ONC_ANALYTICS.MABI_MX_ACT_TBL_AML_EXACT_DATES_KA_v2 ''')

spark.sql('''
CREATE TABLE Z_ABV_CWS_MABI_ONC_ANALYTICS.MABI_MX_ACT_TBL_AML_EXACT_DATES_KA_v2 AS
(       
        SELECT DISTINCT PATIENT_SK AS PATIENT_GID, CLAIM_DATE
        FROM abv_val_ptd_onc_venc_synd.all_claims_combined_fact_vw  
        WHERE PATIENT_SK IN (
          SELECT DISTINCT PATIENT_GID
          FROM {pat_comb_lot}
        )
          AND (SOURCE_TYPE LIKE '%PX%' OR SOURCE_TYPE LIKE '%SX%' OR SOURCE_TYPE LIKE '%DX%')
          --Dx fact is currentlty not in VAL all claims -- 
          union
         SELECT DISTINCT PATIENT_SK AS PATIENT_GID, CLAIM_DATE
        FROM abv_val_ptd_onc_synd.dx_fact_curated_vw  
        WHERE PATIENT_SK IN (
          SELECT DISTINCT PATIENT_GID
          FROM {pat_comb_lot}
        )
)         
          '''.format(pat_comb_lot=pat_comb_lot))


In [ ]:
spark.sql(''' DROP TABLE IF EXISTS Z_ABV_CWS_MABI_ONC_ANALYTICS.MABI_RX_ACT_TBL_AML_EXACT_DATES_KA ''')

spark.sql('''
CREATE TABLE Z_ABV_CWS_MABI_ONC_ANALYTICS.MABI_RX_ACT_TBL_AML_EXACT_DATES_KA AS
(       
        SELECT DISTINCT PATIENT_SK AS PATIENT_GID, CLAIM_DATE
        FROM abv_val_ptd_onc_venc_synd.all_claims_combined_fact_vw  
        WHERE PATIENT_SK IN (
          SELECT DISTINCT PATIENT_GID 
          FROM {pat_comb_lot}
        )
          AND SOURCE_TYPE LIKE '%RX%'
)         
          '''.format(pat_comb_lot=pat_comb_lot))


In [ ]:
spark.sql('''
SELECT COUNT(DISTINCT PATIENT_GID)
FROM Z_ABV_CWS_MABI_ONC_ANALYTICS.MABI_MX_ACT_TBL_AML_EXACT_DATES_KA_v2
''').show()

In [ ]:
spark.sql('''
SELECT COUNT(DISTINCT PATIENT_GID)
FROM Z_ABV_CWS_MABI_ONC_ANALYTICS.MABI_RX_ACT_TBL_AML_EXACT_DATES_KA
''').show()

#### SEMESTER METHODOLOGY (config-driven)

Every activity flag below asks whether the patient had Mx or Rx activity in **every** 6-month period
around an anchor date. `semester.semester_type` in `config/eligibility.yaml` chooses how those periods
are built:

- **calendar**: fixed calendar halves. January-June = S1, July-December = S2 (e.g. 2026-03-03 is
  "2026 - S1", 2026-07-01 is "2026 - S2"), taken from the claim date. The anchor only decides which
  halves are tested.
- **actual**: rolling 6-month windows counted from each patient's own anchor date, so a window can
  start on any day and can cross a year end.

Anchors: DX lookback = FIRST_DX; AML journey = FIRST_TX to LAST_TX; VEN journey = VEN_START to
VEN_END; VEN lookback = VEN_START.

`semester_type` can be one value for every flag, or a mapping per flag group
(`dx_lookback`, `journey_continuity`, `ven_lookback`). The shipped default (actual / actual /
calendar) reproduces the original notebook exactly.

The window logic lives in `config/semester.py`, which every flag section calls. The output tables
keep their original schema (one `PATIENT_GID` column). The final eligibility flags step is unchanged.

In [ ]:
# Load the semester methodology from config/eligibility.yaml (path relative to scripts/Notebooks/).
import sys
sys.path.append('../../config')
from load_config import load_config
import semester as sem

elig_cfg = load_config('eligibility.yaml')
semester_types = sem.resolve_semester_types(elig_cfg.get('semester'))   # raises on an unsupported value
print('Semester methodology per flag group:', semester_types)

MX_ACT = 'Z_ABV_CWS_MABI_ONC_ANALYTICS.MABI_MX_ACT_TBL_AML_EXACT_DATES_KA_v2'   # exact-date Mx activity (cell above)
RX_ACT = 'Z_ABV_CWS_MABI_ONC_ANALYTICS.MABI_RX_ACT_TBL_AML_EXACT_DATES_KA'      # exact-date Rx activity (cell above)

# Semester-level activity the calendar VEN lookback has always read (SEMESTERLY frequency tables).
# It is used only in calendar mode; actual mode needs exact claim dates, so it uses MX_ACT / RX_ACT.
VEN_MX_FREQ = {'semester_select': '''
    SELECT DISTINCT PATIENT_SK AS PATIENT_GID, TO_DATE(FREQUENCY_START_DATE) AS PERIOD_START
    FROM abv_val_ptd_synd_work.mapping_mx_activity_freq_tbl
    WHERE source_flag = 'SHA_PTD' AND source_market = 'ONC'
      AND UPPER(FREQUENCY) = 'SEMESTERLY' AND `OFFSET` = '0' '''}
VEN_RX_FREQ = {'semester_select': '''
    SELECT DISTINCT PATIENT_SK AS PATIENT_GID, TO_DATE(FREQUENCY_START_DATE) AS PERIOD_START
    FROM abv_val_ptd_synd_work.mapping_rx_activity_freq_tbl
    WHERE source_flag = 'SHA_PTD' AND source_market = 'ONC'
      AND UPPER(FREQUENCY) = 'SEMESTERLY' AND `OFFSET` = '0'
      AND PATIENT_SK IN (SELECT DISTINCT PATIENT_GID FROM {pat_fst_dx_tx}) '''.format(pat_fst_dx_tx=pat_fst_dx_tx)}


def build_flag(view_name, group, anchor_select, activity_tables, calendar_activity=None, calendar_spine=None):
    '''Run one eligibility check with the configured methodology and register it as a temp view.'''
    stype = semester_types[group]
    query = sem.sql_flag_check(group, stype, anchor_select, activity_tables, exc_dt=exc_dt, cfg=elig_cfg,
                               calendar_activity=calendar_activity, calendar_spine=calendar_spine)
    df = spark.sql(query)
    df.createOrReplaceTempView(view_name)
    print(f'{view_name}: {group} ({stype})')
    return df

# ELIGIBILITY APPLICATION

#### FIRST DX TX TABLE

In [ ]:
fst_dx_tx_tbl = spark.sql('''
SELECT *
FROM
(
    SELECT A.PATIENT_GID,
           B.FIRST_DX,
           A.FIRST_TX,
           A.LAST_TX,
           C.VEN_START,
           C.VEN_END
    FROM
    (
        SELECT PATIENT_GID,
               MIN(TO_DATE(regimen_start_date)) AS FIRST_TX,
               MAX(TO_DATE(regimen_end_date)) AS LAST_TX
        FROM {pat_comb_lot}
        GROUP BY PATIENT_GID
    ) A
    LEFT JOIN
    (
        SELECT PATIENT_SK AS PATIENT_GID,
               MIN(TO_DATE(CLAIM_DATE)) AS FIRST_DX
        FROM abv_val_ptd_onc_synd.dx_fact_curated_vw
        WHERE SOURCE_DIAGNOSIS_CODE_1 IN 
        (
            '205', '205.0', '205.00', '205.01', '205.02', 'C92', 'C92.9', 'C92.0', 'C92.00', 'C92.01', 
            'C92.02', 'C92.5', 'C92.50', 'C92.51', 'C92.52', 'C92.6', 'C92.60', 'C92.61', 'C92.62', 'C92.90', 
            'C92.A', 'C92.A0', 'C92.A1', 'C92.A2', 'C92.Z0', 'C93', 'C93.0', 'C93.00', 'C93.01', 'C93.02', 
            'C94.0', 'C94.00', 'C94.01', 'C94.2', 'C94.20', 'C94.21', 'C94.22'
        )
        --AND CLAIM_DATE >= '2019-12-01'
        --AND CLAIM_DATE <= '2026-03-06'
        GROUP BY PATIENT_SK
    ) B
    ON A.PATIENT_GID = B.PATIENT_GID
    LEFT JOIN
    (
        SELECT PATIENT_GID, 
               MIN(EPISODE_START_DATE_DRVD) AS VEN_START,
               MAX(EPISODE_END_DATE1_DRVD) AS VEN_END
        FROM {episode_tbl_name}     
        WHERE FINAL_PRODUCT_NAME = 'VENCLEXTA'   
        GROUP BY PATIENT_GID            
    ) C  
    ON A.PATIENT_GID = C.PATIENT_GID                    
) A
'''.format(episode_tbl_name=episode_tbl_name, pat_comb_lot=pat_comb_lot))

fst_dx_tx_tbl.createOrReplaceTempView('fst_dx_tx_tbl')


In [ ]:
spark.sql('''
DROP TABLE IF EXISTS {pat_fst_dx_tx}
'''.format(pat_fst_dx_tx=pat_fst_dx_tx))

In [ ]:
spark.sql('''
CREATE TABLE {pat_fst_dx_tx} AS
(
    SELECT *
    from fst_dx_tx_tbl
)
'''.format(pat_fst_dx_tx=pat_fst_dx_tx))

In [ ]:
spark.sql('''
SELECT patient_gid, FIRST_DX, LAST_TX 
FROM {pat_fst_dx_tx}
where FIRST_DX is NULL OR LAST_TX IS NULL
'''.format(pat_fst_dx_tx=pat_fst_dx_tx)).toPandas() # should return no rows

In [ ]:
spark.sql('''
SELECT COUNT(DISTINCT PATIENT_GID)
FROM {pat_fst_dx_tx}
'''.format(pat_fst_dx_tx=pat_fst_dx_tx)).toPandas()

#### DX (MX or RX) LOOKBACK - FROM AML TX START

In [ ]:
# DX lookback, anchored on FIRST_DX (DX_LB_ELIG_FLG).
#   actual   (default): activity (Mx or Rx) in [FIRST_DX-6m, FIRST_DX) AND in [FIRST_DX-12m, FIRST_DX-6m).
#   calendar: activity in every calendar semester from semester(FIRST_DX-12m) to semester(FIRST_DX).
build_flag('elig_pats', 'dx_lookback',
           anchor_select='SELECT DISTINCT PATIENT_GID, FIRST_DX AS ANCHOR FROM {pat_fst_dx_tx}'.format(pat_fst_dx_tx=pat_fst_dx_tx),
           activity_tables=[MX_ACT, RX_ACT])

In [ ]:
dx_lookback_elig

In [ ]:
spark.sql('''
DROP TABLE IF EXISTS {dx_lookback_elig}
'''.format(dx_lookback_elig=dx_lookback_elig))

In [ ]:
spark.sql('''
CREATE TABLE {dx_lookback_elig} AS
(
    SELECT *
    FROM elig_pats
)
'''.format(dx_lookback_elig=dx_lookback_elig))

In [ ]:
spark.sql('''
SELECT COUNT(DISTINCT PATIENT_GID)
FROM {dx_lookback_elig}
'''.format(dx_lookback_elig=dx_lookback_elig)).show()

#### FIRST AND LAST TX (MX) - OVER AML JOURNEY

In [ ]:
# AML journey continuity on Mx claims, FIRST_TX -> LAST_TX (TX_TXL_MX_ELIG_FLG).
#   actual   (default): every rolling window (FIRST_TX+(n-1)*6m, FIRST_TX+n*6m] up to LAST_TX has a claim;
#            a journey shorter than one window passes automatically.
#   calendar: every calendar semester from semester(FIRST_TX) to semester(LAST_TX) has a claim;
#            a journey inside one semester passes automatically.
build_flag('elig_pats_tx_txl_mx', 'journey_continuity',
           anchor_select='SELECT DISTINCT PATIENT_GID, FIRST_TX AS P_START, LAST_TX AS P_END FROM fst_dx_tx_tbl',
           activity_tables=[MX_ACT])

In [ ]:
spark.sql('''
          select count(distinct patient_gid)
          from elig_pats_tx_txl_mx
        
          ''').show()

In [ ]:
tx_txl_mx_elig

In [ ]:
spark.sql('''
DROP TABLE IF EXISTS {tx_txl_mx_elig}
'''.format(tx_txl_mx_elig=tx_txl_mx_elig))

In [ ]:
spark.sql('''
CREATE TABLE {tx_txl_mx_elig} AS
(
    SELECT *
    FROM elig_pats_tx_txl_mx
)
'''.format(tx_txl_mx_elig=tx_txl_mx_elig))

In [ ]:
spark.sql('''
SELECT COUNT(DISTINCT PATIENT_GID)
FROM {tx_txl_mx_elig}
'''.format(tx_txl_mx_elig=tx_txl_mx_elig)).show()

#### FIRST TX, LAST TX (RX) - OVER AML JOURNEY

In [ ]:
# AML journey continuity on Rx claims, FIRST_TX -> LAST_TX (TX_TXL_RX_ELIG_FLG). Same rules as the Mx check.
build_flag('elig_pats_tx_txl_rx', 'journey_continuity',
           anchor_select='SELECT DISTINCT PATIENT_GID, FIRST_TX AS P_START, LAST_TX AS P_END FROM fst_dx_tx_tbl',
           activity_tables=[RX_ACT])

In [ ]:
spark.sql('''
          select count(distinct patient_gid)
          from elig_pats_tx_txl_rx
        
          ''').show()

In [ ]:
tx_txl_rx_elig

In [ ]:
spark.sql('''
DROP TABLE IF EXISTS {tx_txl_rx_elig}
'''.format(tx_txl_rx_elig=tx_txl_rx_elig))

In [ ]:
spark.sql('''
CREATE TABLE {tx_txl_rx_elig} AS
(
    SELECT *
    FROM elig_pats_tx_txl_rx
)
'''.format(tx_txl_rx_elig=tx_txl_rx_elig))

In [ ]:
spark.sql('''
SELECT COUNT(DISTINCT PATIENT_GID)
FROM {tx_txl_rx_elig}
'''.format(tx_txl_rx_elig=tx_txl_rx_elig)).show()

#### TX (MX) LOOKBACK - FROM VEN START

In [ ]:
# VEN lookback anchor: VEN_START. Patients without Venclexta have no anchor and fail the lookback (as before).
VEN_LOOKBACK_ANCHOR = 'SELECT DISTINCT PATIENT_GID, VEN_START AS ANCHOR FROM fst_dx_tx_tbl'

# Calendar mode only: the original query tested only semesters present in VENC_AML 'RX FACT' claims.
VEN_SEMESTER_SPINE = '''
    SELECT DISTINCT {sem_start} AS PERIOD_START
    FROM abv_val_ptd_onc_venc_synd.all_claims_combined_fact_vw
    WHERE source_flag = 'SHA_PTD' AND source_market = 'ONC'
      AND market_code = 'VENC_AML' AND source_type = 'RX FACT' '''.format(
    sem_start=sem.sql_calendar_semester_start('TO_DATE(CLAIM_DATE)'))

In [ ]:
# VEN lookback (calendar, as before): semesters are taken from the VENC_AML 'RX FACT' semester spine,
# from semester(VEN_START - 12m) to semester(VEN_START), capped at exc_dt; activity comes from the
# SEMESTERLY frequency tables. In actual mode: two rolling windows [VEN_START-12m, VEN_START-6m) and
# [VEN_START-6m, VEN_START) on exact claim dates.

build_flag('elig_pats_tx_ven', 'ven_lookback',
           anchor_select=VEN_LOOKBACK_ANCHOR,
           activity_tables=[MX_ACT], calendar_activity=[VEN_MX_FREQ], calendar_spine=VEN_SEMESTER_SPINE)

In [ ]:
spark.sql(''' DROP TABLE IF EXISTS {tx_lookback_elig_ven} '''.format(tx_lookback_elig_ven=tx_lookback_elig_ven))

In [ ]:
spark.sql('''
CREATE TABLE {tx_lookback_elig_ven} AS
(
    SELECT *
    FROM elig_pats_tx_ven
)
'''.format(tx_lookback_elig_ven=tx_lookback_elig_ven))

#### TX (RX) LOOKBACK - FROM VEN START

In [ ]:
# VEN lookback on Rx activity (TX_RX_LB_ELIG_VEN_FLG). Same rules as the Mx check.
build_flag('elig_pats_tx_rx_ven', 'ven_lookback',
           anchor_select=VEN_LOOKBACK_ANCHOR,
           activity_tables=[RX_ACT], calendar_activity=[VEN_RX_FREQ], calendar_spine=VEN_SEMESTER_SPINE)

In [ ]:
spark.sql(''' DROP TABLE IF EXISTS {tx_lookback_elig_rx_ven} '''.format(tx_lookback_elig_rx_ven=tx_lookback_elig_rx_ven))

In [ ]:
spark.sql('''
CREATE TABLE {tx_lookback_elig_rx_ven} AS
(
    SELECT *
    FROM elig_pats_tx_rx_ven
)
'''.format(tx_lookback_elig_rx_ven=tx_lookback_elig_rx_ven))

#### FIRST AND LAST TX (MX) - OVER VEN JOURNEY

In [ ]:
# VEN journey continuity on Mx claims, VEN_START -> VEN_END (TX_TXL_MX_ELIG_VEN_FLG). Same rules as the AML
# journey. Patients without Venclexta (no VEN_START) pass, as before.
build_flag('elig_pats_tx_txl_mx_ven', 'journey_continuity',
           anchor_select='SELECT DISTINCT PATIENT_GID, VEN_START AS P_START, VEN_END AS P_END FROM fst_dx_tx_tbl',
           activity_tables=[MX_ACT])

In [ ]:
spark.sql('''
          select count(distinct patient_gid)
          from elig_pats_tx_txl_mx_ven
        
          ''').show()

In [ ]:
spark.sql(''' DROP TABLE IF EXISTS {tx_txl_mx_elig_ven} '''.format(tx_txl_mx_elig_ven=tx_txl_mx_elig_ven))

In [ ]:
spark.sql('''
CREATE TABLE {tx_txl_mx_elig_ven} AS
(
    SELECT *
    FROM elig_pats_tx_txl_mx_ven
)
'''.format(tx_txl_mx_elig_ven=tx_txl_mx_elig_ven))

#### FIRST TX, LAST TX (RX) - OVER VEN JOURNEY

In [ ]:
# VEN journey continuity on Rx claims, VEN_START -> VEN_END (TX_TXL_RX_ELIG_VEN_FLG).
build_flag('elig_pats_tx_txl_rx_ven', 'journey_continuity',
           anchor_select='SELECT DISTINCT PATIENT_GID, VEN_START AS P_START, VEN_END AS P_END FROM fst_dx_tx_tbl',
           activity_tables=[RX_ACT])

In [ ]:
spark.sql('''
          select count(distinct patient_gid)
          from elig_pats_tx_txl_rx_ven
        
          ''').show()

In [ ]:
spark.sql(''' DROP TABLE IF EXISTS {tx_txl_rx_elig_ven} '''.format(tx_txl_rx_elig_ven=tx_txl_rx_elig_ven))

In [ ]:
spark.sql('''
CREATE TABLE {tx_txl_rx_elig_ven} AS
(
    SELECT *
    FROM elig_pats_tx_txl_rx_ven
)
'''.format(tx_txl_rx_elig_ven=tx_txl_rx_elig_ven))

#### FINAL ELIGIBILITY FLAGS

In [ ]:


final_elig_flags = spark.sql('''
SELECT PATS.*, 
CASE WHEN DX_LB_ELIG.ELIG_FLG = 1 THEN 1 ELSE 0 END AS DX_LB_ELIG_FLG,
--CASE WHEN TX_MX_LB_ELIG.ELIG_FLG_TX_MX = 1 THEN 1 ELSE 0 END AS TX_MX_LB_ELIG_FLG,
--CASE WHEN TX_RX_LB_ELIG.ELIG_FLG_TX_RX = 1 THEN 1 ELSE 0 END AS TX_RX_LB_ELIG_FLG,
CASE WHEN TX_TXL_M_ELIG.ELIG_FLG_TX_TXL_MX = 1 THEN 1 ELSE 0 END AS TX_TXL_MX_ELIG_FLG,
CASE WHEN TX_TXL_R_ELIG.ELIG_FLG_TX_TXL_RX = 1 THEN 1 ELSE 0 END AS TX_TXL_RX_ELIG_FLG,

CASE WHEN TX_MX_LB_ELIG_VEN.ELIG_FLG_TX_MX_VEN = 1 THEN 1 ELSE 0 END AS TX_MX_LB_ELIG_VEN_FLG,
CASE WHEN TX_RX_LB_ELIG_VEN.ELIG_FLG_TX_RX_VEN = 1 THEN 1 ELSE 0 END AS TX_RX_LB_ELIG_VEN_FLG,
CASE WHEN TX_TXL_M_ELIG_VEN.ELIG_FLG_TX_TXL_MX_VEN = 1 THEN 1 ELSE 0 END AS TX_TXL_MX_ELIG_VEN_FLG,
CASE WHEN TX_TXL_R_ELIG_VEN.ELIG_FLG_TX_TXL_RX_VEN = 1 THEN 1 ELSE 0 END AS TX_TXL_RX_ELIG_VEN_FLG,

CASE 
	WHEN DATEDIFF(to_date(FIRST_TX),to_date(FIRST_DX)) >= 0 AND DATEDIFF(to_date(FIRST_TX),to_date(FIRST_DX)) <= 60 THEN 1 
	ELSE 0 
END AS DX_TX_DIFF_FLG,
CASE WHEN FIRST_TX >= FIRST_DX THEN 1 ELSE 0 END AS TX_POST_DX_FLG,
TYPE.TYPE AS PATIENT_COHORT,
TYPE_UPDATED.PATIENT_INTENSITY_UPDATED AS PATIENT_COHORT_UPDATED,                          
CASE WHEN SCT_PX.SCT_PX_DATE >= FIRST_DX THEN 1 ELSE 0 END AS SCT_PX_FLG,
CASE WHEN SCT_DX.FIRST_SCT_DX >= FIRST_DX THEN 1 ELSE 0 END AS SCT_DX_FLG,
CASE WHEN SCT_PX.SCT_PX_DATE >= FIRST_DX OR SCT_DX.FIRST_SCT_DX >= FIRST_DX THEN 1 ELSE 0 END AS SCT_DX_PX_FLG,
COALESCE(ARSENIC_CHK.ARSENIC_FLG,1) AS ARSENIC_FLG --SHOULDN'T BE NULL FOR ANY PATIENT
FROM
(
	SELECT DISTINCT PATIENT_GID, FIRST_DX, FIRST_TX, LAST_TX
	FROM {pat_fst_dx_tx}
)PATS
---------AML TX ELIG FLAGS START----------------------
LEFT JOIN
(
	SELECT DISTINCT PATIENT_GID, 1 AS ELIG_FLG
	FROM {dx_lookback_elig}
)DX_LB_ELIG
ON PATS.PATIENT_GID = DX_LB_ELIG.PATIENT_GID
--LEFT JOIN
--(
--	SELECT DISTINCT PATIENT_GID, 1 AS ELIG_FLG_TX_MX
--	FROM tx_lookback_elig
--)TX_MX_LB_ELIG
--ON PATS.PATIENT_GID = TX_MX_LB_ELIG.PATIENT_GID
--LEFT JOIN
--(
--	SELECT DISTINCT PATIENT_GID, 1 AS ELIG_FLG_TX_RX
--	FROM tx_lookback_elig_rx
---)TX_RX_LB_ELIG
--ON PATS.PATIENT_GID = TX_RX_LB_ELIG.PATIENT_GID
LEFT JOIN
(
	SELECT DISTINCT PATIENT_GID, 1 AS ELIG_FLG_TX_TXL_MX
	FROM {tx_txl_mx_elig}
)TX_TXL_M_ELIG
ON PATS.PATIENT_GID = TX_TXL_M_ELIG.PATIENT_GID
LEFT JOIN
(
	SELECT DISTINCT PATIENT_GID, 1 AS ELIG_FLG_TX_TXL_RX
	FROM {tx_txl_rx_elig}
)TX_TXL_R_ELIG
ON PATS.PATIENT_GID = TX_TXL_R_ELIG.PATIENT_GID
---------AML TX ELIG FLAGS END----------------------
---------VEN ELIG FLAGS START-----------------------
LEFT JOIN
(
	SELECT DISTINCT PATIENT_GID, 1 AS ELIG_FLG_TX_MX_VEN
	FROM {tx_lookback_elig_ven}
)TX_MX_LB_ELIG_VEN
ON PATS.PATIENT_GID = TX_MX_LB_ELIG_VEN.PATIENT_GID
LEFT JOIN
(
	SELECT DISTINCT PATIENT_GID, 1 AS ELIG_FLG_TX_RX_VEN
	FROM {tx_lookback_elig_rx_ven}
)TX_RX_LB_ELIG_VEN
ON PATS.PATIENT_GID = TX_RX_LB_ELIG_VEN.PATIENT_GID
LEFT JOIN
(
	SELECT DISTINCT PATIENT_GID, 1 AS ELIG_FLG_TX_TXL_MX_VEN
	FROM {tx_txl_mx_elig_ven}
)TX_TXL_M_ELIG_VEN
ON PATS.PATIENT_GID = TX_TXL_M_ELIG_VEN.PATIENT_GID
LEFT JOIN
(
	SELECT DISTINCT PATIENT_GID, 1 AS ELIG_FLG_TX_TXL_RX_VEN
	FROM {tx_txl_rx_elig_ven}
)TX_TXL_R_ELIG_VEN
ON PATS.PATIENT_GID = TX_TXL_R_ELIG_VEN.PATIENT_GID


---------VEN ELIG FLAGS END---------------------------
LEFT JOIN
(
	SELECT PATIENT_GID, MIN(ARSENIC_FLG_INT) AS ARSENIC_FLG
	FROM
	(
		SELECT PATIENT_GID,
		CASE WHEN REGIMEN LIKE '%ARSENIC_TRIOXIDE%' THEN 0 ELSE 1 END AS ARSENIC_FLG_INT
		FROM {pat_comb_lot}
	)A
	GROUP BY 1
)ARSENIC_CHK
ON PATS.PATIENT_GID = ARSENIC_CHK.PATIENT_GID

---------PATIENT TYPE---------------------------
LEFT JOIN
(
	SELECT DISTINCT PATIENT_GID, TYPE
	FROM {pat_comb_lot}
)TYPE
ON PATS.PATIENT_GID = TYPE.PATIENT_GID

LEFT JOIN 
(
           SELECT DISTINCT PATIENT_GID, PATIENT_INTENSITY_UPDATED
           FROM {rule_b_tbl}       
)TYPE_UPDATED
ON PATS.PATIENT_GID = TYPE_UPDATED.PATIENT_GID
                                                 
                             
---------SCT DATES MAPPING---------------------------

LEFT JOIN
(
	select PATIENT_GID, MIN(TO_DATE(SCT_PX_DATE)) AS SCT_PX_DATE
	from 
	(

		SELECT PATIENT_SK AS PATIENT_GID, MIN(TO_DATE(CLAIM_DATE)) AS SCT_PX_DATE
		FROM abv_val_ptd_onc_synd.Sx_fact_curated_vw
		WHERE SURGICAL_code IN 
		(
			'38240','38241','38243','38242','30230G1','30230G2','30230G3','30230G4','30230U2','30230U3',
			'30230U4','30230Y1','30230Y2','30230Y3','30230Y4','30233G1','30233G2','30233G3','30233G4',
			'30233Y1','30233Y2','30233Y3','30233Y4','30240G1','30240G2','30240G3','30240G4','30240Y1',
			'30240Y2','30240Y3','30240Y4','30243G1','30243G2','30243G3','30243G4','30243Y1','30243Y2',
			'30243Y3','30243Y4','30250G1','30250Y1','30253G1','30253Y1','30260G1','30260Y1','30263G1',
			'30263Y1','4102','4103','4105','4108'
		)
		and source_flag  =  'SHA_PTD'
		and source_market  =  'ONC'
			
		GROUP BY 1

		union

		SELECT PATIENT_SK AS PATIENT_GID, MIN(TO_DATE(CLAIM_DATE)) AS SCT_PX_DATE
		FROM abv_val_ptd_onc_synd.px_fact_curated_vw
		WHERE procedure_code IN 
		(
			'38240','38241','38243','38242','30230G1','30230G2','30230G3','30230G4','30230U2','30230U3',
			'30230U4','30230Y1','30230Y2','30230Y3','30230Y4','30233G1','30233G2','30233G3','30233G4',
			'30233Y1','30233Y2','30233Y3','30233Y4','30240G1','30240G2','30240G3','30240G4','30240Y1',
			'30240Y2','30240Y3','30240Y4','30243G1','30243G2','30243G3','30243G4','30243Y1','30243Y2',
			'30243Y3','30243Y4','30250G1','30250Y1','30253G1','30253Y1','30260G1','30260Y1','30263G1',
			'30263Y1','4102','4103','4105','4108'
		)
		and source_flag  =  'SHA_PTD'
		and source_market  =  'ONC'
			
		GROUP BY 1

	)

	GROUP BY 1
)SCT_PX
ON PATS.PATIENT_GID = SCT_PX.PATIENT_GID

LEFT JOIN
(
	SELECT PATIENT_SK AS PATIENT_GID, MIN(CLAIM_DATE) AS FIRST_SCT_DX 
	FROM abv_val_ptd_onc_synd.dx_fact_curated_vw

	WHERE source_diagnosis_code_1 IN 
	(
	'Z94.84', 'V42.82'
	)
	GROUP BY 1
)SCT_DX
ON PATS.PATIENT_GID = SCT_DX.PATIENT_GID

'''.format(dx_lookback_elig=dx_lookback_elig, 
tx_txl_mx_elig=tx_txl_mx_elig, tx_txl_rx_elig=tx_txl_rx_elig,pat_fst_dx_tx=pat_fst_dx_tx,
tx_lookback_elig_ven=tx_lookback_elig_ven,tx_lookback_elig_rx_ven=tx_lookback_elig_rx_ven,tx_txl_mx_elig_ven=tx_txl_mx_elig_ven,
tx_txl_rx_elig_ven=tx_txl_rx_elig_ven, pat_comb_lot=pat_comb_lot, rule_b_tbl=rule_b_tbl))

final_elig_flags.createOrReplaceTempView('final_elig_flags')

In [ ]:
spark.sql('''DROP TABLE IF EXISTS {elig_flags_final}'''.format(elig_flags_final=elig_flags_final))

In [ ]:
elig_flags_final

In [ ]:
spark.sql('''
CREATE TABLE {elig_flags_final}
(
    SELECT *
    FROM final_elig_flags
)
'''.format(elig_flags_final=elig_flags_final))

In [ ]:
spark.sql('''
select * from {elig_flags_final} limit 5
'''.format(elig_flags_final=elig_flags_final)).toPandas()